# External validation across independent bulk substantia nigra cohorts

A companion to `pd-lcm-rf-external` (GSE7621 only), which is left as it is. This notebook adds:

1. **More cohorts** - GSE7621, GSE20292, GSE20163, GSE20164, GSE8397, GSE49036 (arrays) and GSE114517, GSE168496
   (RNA-seq); GSE49036 also gives early-stage donors (incidental Lewy body disease, Braak 1-4).
2. **Donor overlap** - where donor IDs are public they are matched against the discovery studies (and the matches
   checked for diagnosis and sex); shared donors are removed. Every cohort is also scored by a **strictly
   independent model** - the same locked pipeline, retrained without any discovery study that could come from the
   same brain bank.
3. **Calls, not only ranking** - accuracy, sensitivity and specificity at two thresholds fixed on the discovery
   people before any external cohort is seen: 0.5, and the out-of-bag optimum of each forest.

**The model was locked before any of these cohorts was examined.** The core classifier (within-person ranks ->
PCA 30 -> Random Forest) was chosen by a sweep that used only the 63 discovery people (`pd-lcm-rf-confirm`). Nothing
is fitted on an external cohort: each goes through the same label-free steps as discovery (genes z-scored within the
cohort, then ranked within each person) and is scored once.

In [ ]:
import os, io, re, gzip, glob, json, time, tarfile, urllib.request, warnings
from pathlib import Path
import numpy as np, pandas as pd
import joblib
from scipy.stats import rankdata, spearmanr, binomtest
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, balanced_accuracy_score
warnings.filterwarnings("ignore")
OUT = Path("/kaggle/working"); GEO = OUT / "geo"; GEO.mkdir(exist_ok=True)
def find_any(pattern, key):
    hits = sorted((h for h in glob.glob(f"/kaggle/input/**/{pattern}", recursive=True) if key in h), key=len)
    if not hits:
        raise FileNotFoundError(f"{pattern} ({key})")
    return hits[0]
t0 = time.time()
def log(m): print(f"[{time.time() - t0:5.0f}s] {m}", flush=True)
pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 70); pd.set_option("display.max_columns", 40)

In [ ]:
DISC = json.loads(r'''{"GSE20141": {"C-1074p-SNc": [0, 0], "C-1271p-SNc": [0, 1], "C-2829-SNc": [0, 0], "C-3132p-SNc": [0, 0], "C-3397-SNc": [0, 1], "C-3543-SNc": [0, 1], "C-3603-SNc": [0, 1], "C-5220-SNc": [0, 0], "PD-1364-SNc": [1, 0], "PD-1401-SNc": [1, 0], "PD-1647-SNc": [1, 1], "PD-2515p-SNc": [1, 1], "PD-2525-SNc": [1, 1], "PD-3769p-SNc": [1, 0], "PD-3790p-SNc": [1, 1], "PD-3803p-SNc": [1, 0], "PD-5138-SNc": [1, 0], "PD-5476-SNc": [1, 0]}, "GSE182622": {"C-04-52": [0, 0], "C-06-57": [0, 0], "C-07-28": [0, 0], "C-09-50": [0, 0], "C-10-39": [0, 0], "C-12-44": [0, 1], "C-14-42": [0, 0], "C-15-46": [0, 1], "C-15-60": [0, 0], "C-15-78": [0, 1], "PD-03-43": [1, 0], "PD-03-45": [1, 1], "PD-06-44": [1, 0], "PD-10-27": [1, 1], "PD-10-83": [1, 0], "PD-11-110": [1, 1], "PD-12-22": [1, 0], "PD-12-33": [1, 0], "PD-12-55": [1, 0], "PD-13-29": [1, 0], "PD-95-19": [1, 0], "PD-96-36": [1, 0]}}''')   # discovery donor -> [PD, female (from expression)]

## 1. Discovery: the frozen core model, its pre-specified thresholds, the strictly independent models

In [ ]:
cz = np.load(find_any("core_data.npz", "rf-core"), allow_pickle=True)
X, XR, y, DS = cz["X"], cz["XR"], cz["y"].astype(int), cz["ds"].astype(str)
GENES = [str(g) for g in cz["genes"]]
CORE = joblib.load(find_any("core_model.joblib", "rf-core"))
SYM = pd.read_csv(find_any("15_gene_symbol_map.csv", "rf-core")).set_index("gene")["symbol"].to_dict()
PANEL = pd.read_csv(find_any("04_boruta_selected_genes.csv", "boruta-panel")).gene.tolist()
DE = pd.read_csv(find_any("03_de_results_full.csv", "boruta-panel")).set_index("gene")
PI = [GENES.index(g) for g in PANEL]

def oob_threshold(s, yt):
    """The accuracy cut-off on out-of-bag scores of the training people (the rule used in every discovery fold)."""
    u = np.unique(np.round(s, 6)); cuts = np.concatenate([[-np.inf], (u[:-1] + u[1:]) / 2, [np.inf]])
    acc = [accuracy_score(yt, (s > c).astype(int)) for c in cuts]
    best = np.flatnonzero(np.isclose(acc, max(acc)))
    return float(cuts[best[len(best) // 2]])
def fit_core(mask):
    """The locked pipeline (within-person ranks -> PCA 30 -> Random Forest) on a subset of the discovery people."""
    p = PCA(min(30, int(mask.sum()) - 1), random_state=42).fit(XR[mask])        # 30 components, fewer only if too few people
    rf = RandomForestClassifier(n_estimators=1000, max_features=0.5, min_samples_leaf=1, class_weight="balanced",
                                oob_score=True, random_state=42, n_jobs=-1).fit(p.transform(XR[mask]), y[mask])
    return {"pca": p, "rf": rf, "thr": oob_threshold(rf.oob_decision_function_[:, 1], y[mask]), "n": int(mask.sum()),
            "components": p.n_components_}
CORE["thr"], CORE["n"] = oob_threshold(CORE["rf"].oob_decision_function_[:, 1], y), len(y)
# discovery studies that could share a brain bank with an external cohort
RELATED = {"harvard": ["GSE20141", "GSE24378"], "nbb": ["GSE182622"]}
STRICT = {k: fit_core(~np.isin(DS, v)) for k, v in RELATED.items()}
STRICT["none"] = CORE
PANEL_RF = [RandomForestClassifier(n_estimators=1000, max_features="sqrt", class_weight="balanced", oob_score=True,
                                   random_state=42 + k, n_jobs=-1).fit(X[:, PI], y) for k in range(5)]
PANEL_THR = oob_threshold(np.mean([m.oob_decision_function_[:, 1] for m in PANEL_RF], axis=0), y)
log(f"discovery {len(y)} people x {len(GENES):,} genes")
log("strictly independent models: " + ", ".join(f"without {'+'.join(v)} ({STRICT[k]['n']} people, {STRICT[k]['components']} components)" for k, v in RELATED.items()))

# the panel, made strictly independent the same way: Boruta (the reported settings) re-run without the related studies
for _a, _t in [("float", float), ("int", int), ("bool", bool), ("object", object)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
try:
    from boruta import BorutaPy
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Boruta==0.4.3"], check=True)
    from boruta import BorutaPy
def fit_panel(mask):
    """Boruta (500 trees, perc 99, 100 iterations) on a subset of the discovery people, then five forests on its genes."""
    idx = np.flatnonzero(mask)
    b = BorutaPy(RandomForestClassifier(max_features="sqrt", class_weight="balanced", n_jobs=-1), n_estimators=500, perc=99,
                 alpha=0.05, two_step=True, max_iter=100, random_state=42, verbose=0).fit(X[idx], y[idx])
    genes = list(np.flatnonzero(b.support_)) or list(np.flatnonzero(b.support_ | b.support_weak_))
    rfs = [RandomForestClassifier(n_estimators=1000, max_features="sqrt", class_weight="balanced", oob_score=True,
                                  random_state=42 + k, n_jobs=-1).fit(X[idx][:, genes], y[idx]) for k in range(5)]
    return {"genes": genes, "rfs": rfs, "n": len(idx),
            "thr": oob_threshold(np.mean([m.oob_decision_function_[:, 1] for m in rfs], axis=0), y[idx])}
PANELS = {"none": {"genes": PI, "rfs": PANEL_RF, "n": len(y), "thr": PANEL_THR}}
for k, v in RELATED.items():
    PANELS[k] = fit_panel(~np.isin(DS, v))
    log(f"strict panel without {'+'.join(v)}: {len(PANELS[k]['genes'])} genes, {len(set(PANELS[k]['genes']) & set(PI))} of them "
        f"in the reported 30-gene panel: " + ", ".join(SYM.get(GENES[j], GENES[j]) for j in PANELS[k]["genes"]))
THRESHOLDS = {"core": CORE["thr"], **{f"core_strict_{k}": STRICT[k]["thr"] for k in RELATED},
              "panel": PANEL_THR, **{f"panel_strict_{k}": PANELS[k]["thr"] for k in RELATED}}
log("pre-specified out-of-bag thresholds: " + ", ".join(f"{k} {v:.3f}" for k, v in THRESHOLDS.items()))

## 2. Download and parse the cohorts (on Kaggle, from GEO)

In [ ]:
def geo_url(acc, sub, f): return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{acc[:-3]}nnn/{acc}/{sub}/{f}"
def fetch(acc, sub, f):
    dest = GEO / f
    for k in range(6):
        try:
            if not dest.exists():
                urllib.request.urlretrieve(geo_url(acc, sub, f), dest)
            return dest
        except Exception as exc:
            print("  retry", k + 1, type(exc).__name__); dest.unlink(missing_ok=True); time.sleep(10)
    raise RuntimeError(f"could not download {f}")
def read_matrix(path):
    op = gzip.open if str(path).endswith(".gz") else open
    lines = op(path, "rt", encoding="utf-8", errors="replace").read().split("\n")
    meta = {}
    for l in lines:
        if l.startswith("!Sample_"):
            k = l.split("\t")[0]; meta.setdefault(k, []).append([v.strip().strip('"') for v in l.split("\t")[1:]])
    b0 = next((i for i, l in enumerate(lines) if "!series_matrix_table_begin" in l), None)
    b1 = next((i for i, l in enumerate(lines) if "!series_matrix_table_end" in l), None)
    M = None
    if b0 is not None and b1 - b0 > 2:
        M = pd.read_csv(io.StringIO("\n".join(lines[b0 + 1:b1])), sep="\t", index_col=0).apply(pd.to_numeric, errors="coerce")
    return meta, M
def series_matrix(acc, fname=None): return read_matrix(fetch(acc, "matrix", fname or f"{acc}_series_matrix.txt.gz"))
def chars(meta, *keys):
    """One characteristic across samples, whichever characteristics row holds it (first key found)."""
    out = [None] * len(meta["!Sample_geo_accession"][0])
    for row in meta.get("!Sample_characteristics_ch1", []):
        for i, v in enumerate(row):
            for key in keys:
                if out[i] is None and v.lower().startswith(key.lower() + ":"):
                    out[i] = v.split(":", 1)[1].strip()
    return out
def gconvert(ids, target, batch=2500):
    out = {}; ids = list(ids)
    for s in range(0, len(ids), batch):
        body = {"organism": "hsapiens", "target": target, "query": ids[s:s + batch]}
        for k in range(6):
            try:
                req = urllib.request.Request("https://biit.cs.ut.ee/gprofiler/api/convert/convert/", data=json.dumps(body).encode(),
                                             headers={"Content-Type": "application/json"})
                with urllib.request.urlopen(req, timeout=300) as fh:
                    for rec in json.loads(fh.read())["result"]:
                        c = rec.get("converted")
                        if c and c not in ("None", "N/A"):
                            out.setdefault(rec["incoming"], set()).add(c)
                break
            except Exception as exc:
                print("  g:Convert retry", k + 1, type(exc).__name__); time.sleep(10)
    return out
MARKERS = ["TH", "SLC6A3", "SLC18A2", "DDC", "KCNJ6", "ALDH1A1", "NR4A2", "EN1"]        # dopamine-neuron content
SEXG = ["XIST", "RPS4Y1", "DDX3Y", "KDM5D", "UTY", "EIF1AY"]                             # sex, from expression
def array_genes(M, namespace):
    """Probe matrix -> gene matrix (each gene's highest-mean probe), log2 unless already logged.
    HG-U133A probe sets all exist, under the same IDs, on HG-U133 Plus 2, so both use the Plus 2 mapping."""
    L = np.log2(M.clip(lower=1)) if np.nanmax(M.values) > 50 else M.copy()
    pm = L.mean(1)
    def best(ids):
        conv = gconvert(ids, namespace)
        return {g: max((p for p in ps if p in L.index), key=lambda p: pm[p]) for g, ps in conv.items() if any(p in L.index for p in ps)}
    bg, bm = best(GENES), best(MARKERS + SEXG)
    G = pd.DataFrame({g: L.loc[p].to_numpy(float) for g, p in bg.items()}, index=L.columns).T
    Mk = pd.DataFrame({s: L.loc[p].to_numpy(float) for s, p in bm.items()}, index=L.columns).T
    return G, Mk
def sex_from_expression(Mk):
    """Female = low Y-chromosome genes and high XIST; split at the widest gap of the combined score."""
    z = lambda D: ((D.T - D.T.mean()) / D.T.std(ddof=1).clip(lower=0.05)).T
    yg = [g for g in SEXG[1:] if g in Mk.index]
    s = z(Mk.loc[yg]).mean(0).to_numpy() - (z(Mk.loc[["XIST"]]).mean(0).to_numpy() if "XIST" in Mk.index else 0)
    v = np.sort(s); i = int(np.argmax(np.diff(v)))
    return (s < (v[i] + v[i + 1]) / 2).astype(int)
def meta_sex(vals):
    return [None if v is None else (1 if v.strip().lower().startswith("f") else 0) for v in vals]
COH = {}
def add(name, G, Mk, yv, donors, note, sex=None, stage=None):
    yv, donors = np.asarray(yv, int), np.asarray(donors, str)
    COH[name] = dict(G=G, Mk=Mk, y=yv, donor=donors, note=note, sex_meta=sex if sex is not None else [None] * len(yv),
                     sex_expr=sex_from_expression(Mk), stage=stage if stage is not None else ["PD" if v else "control" for v in yv])
    log(f"{name}: {len(yv)} people ({int((yv == 0).sum())} control, {int(yv.sum())} PD); {G.shape[0]:,}/{len(GENES):,} genes; "
        f"{sum(m in Mk.index for m in MARKERS)}/8 neuron markers  [{note}]")

In [ ]:
# ---------------- GSE7621 (HG-U133 Plus 2) - the same file as pd-lcm-rf-external ----------------
meta, M = read_matrix(find_any("GSE7621_series_matrix.txt*", "externalvalidation2"))
tit = meta["!Sample_title"][0]
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE7621", G, Mk, [0 if "normal" in t.lower() else 1 for t in tit], tit, "Plus 2", sex=meta_sex(chars(meta, "gender", "sex")))

# ---------------- GSE20292 (HG-U133A); donor numbers shared with discovery GSE20141 ----------------
meta, M = series_matrix("GSE20292")
tit = meta["!Sample_title"][0]; dx = chars(meta, "disease state")
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE20292", G, Mk, [0 if d.lower().startswith("control") else 1 for d in dx], [t.split()[0] for t in tit], "U133A",
    sex=meta_sex(chars(meta, "gender")))

# ---------------- GSE20163 (HG-U133A) ----------------
meta, M = series_matrix("GSE20163")
tit = meta["!Sample_title"][0]; src = meta["!Sample_source_name_ch1"][0]
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE20163", G, Mk, [0 if "control" in s.lower() else 1 for s in src], [t.split("_")[0] for t in tit], "U133A")

# ---------------- GSE20164 (HG-U133A) ----------------
meta, M = series_matrix("GSE20164")
tit = meta["!Sample_title"][0]; src = meta["!Sample_source_name_ch1"][0]
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
add("GSE20164", G, Mk, [0 if "control" in s.lower() else 1 for s in src], tit, "U133A", sex=meta_sex(chars(meta, "gender")))

# ---------------- GSE8397 (HG-U133A chip): lateral and medial nigra averaged per case; frontal cortex left out ----------------
meta, M = series_matrix("GSE8397", "GSE8397-GPL96_series_matrix.txt.gz")
tit = meta["!Sample_title"][0]; ag = chars(meta, "age")
keep = [i for i, t in enumerate(tit) if "substantia nigra" in t.lower()]
def case_no(t):
    m = re.search(r"case\s*(\d+)", t, re.I) or re.search(r"(\d+)[^\d]*-\s*[AB]\s*chip", t, re.I) or re.search(r"(\d+)", t)
    return int(m.group(1))
print("   GSE8397 nigra titles:", [tit[i] for i in keep])
case = [("PD" if "parkinson" in tit[i].lower() else "C") + "-" + str(case_no(tit[i])) for i in keep]
sexc = {c: (1 if "gender: f" in (ag[i] or "").lower() else 0) for c, i in zip(case, keep)}
L = M.iloc[:, keep].copy(); L.columns = case
L = L.T.groupby(level=0).mean().T                                          # one profile per person
print("   GSE8397 nigra samples per case:", pd.Series(case).value_counts().value_counts().to_dict())
G, Mk = array_genes(L, "AFFY_HG_U133_PLUS_2")
add("GSE8397", G, Mk, [1 if c.startswith("PD") else 0 for c in L.columns], list(L.columns), "U133A, lateral+medial SN averaged",
    sex=[sexc[c] for c in L.columns])

# ---------------- GSE49036 (Plus 2), Netherlands Brain Bank: control vs PD; ILBD kept for the early-stage test ----------------
meta, M = series_matrix("GSE49036")
tit = meta["!Sample_title"][0]; dx = chars(meta, "disease state"); braak = chars(meta, "braak stage")
G, Mk = array_genes(M, "AFFY_HG_U133_PLUS_2")
lab = np.array([0 if d.lower() == "control" else (1 if "parkinson" in d.lower() else 2) for d in dx])   # 2 = incidental Lewy body
print("   GSE49036 diagnosis x Braak:", pd.crosstab(np.array(dx), np.array(braak)).to_dict())
m = lab < 2
add("GSE49036", G.loc[:, m], Mk.loc[:, m], lab[m], np.array(tit)[m], "Plus 2, NBB")
EARLY_RAW = dict(G=G, Mk=Mk, lab=lab, braak=np.array(braak), donor=np.array(tit))

In [ ]:
# ---------------- GSE114517 (RNA-seq counts per sample; substantia nigra only; PD with dementia) ----------------
meta, _ = series_matrix("GSE114517")
gsm = meta["!Sample_geo_accession"][0]; tit = meta["!Sample_title"][0]
tissue = chars(meta, "tissue"); st = chars(meta, "subject status", "disease state"); sx = chars(meta, "gender", "sex")
sn = [i for i, t in enumerate(tissue) if t and "substantia" in t.lower()]
cols = {}
with tarfile.open(fetch("GSE114517", "suppl", "GSE114517_RAW.tar")) as tf:
    for mem in tf.getmembers():
        g = mem.name.split("_")[0]
        if g in {gsm[i] for i in sn}:
            raw = tf.extractfile(mem).read()
            txt = (gzip.decompress(raw) if mem.name.endswith(".gz") else raw).decode()
            s = pd.read_csv(io.StringIO(txt), sep="\t", header=None, index_col=0)[1]
            s.index = s.index.astype(str).str.split(".").str[0]
            cols[g] = s.groupby(level=0).sum()
C = pd.DataFrame(cols)[[gsm[i] for i in sn]].fillna(0)
C = C[~C.index.str.startswith("__")]
print(f"   GSE114517: {C.shape[1]} nigra libraries, median {np.median(C.sum(0)) / 1e6:.1f} M counted reads")
LC = np.log2(C / C.sum(0) * 1e6 + 1)
SYM2E = {s: sorted(v) for s, v in gconvert(MARKERS + SEXG, "ENSG").items()}
def rna_markers(LG):
    return pd.DataFrame({s: LG.loc[[e for e in es if e in LG.index]].sum(0).to_numpy(float) for s, es in SYM2E.items()
                         if any(e in LG.index for e in es)}, index=LG.columns).T
add("GSE114517", LC.reindex([g for g in GENES if g in LC.index]), rna_markers(LC),
    [0 if "control" in st[i].lower() else 1 for i in sn], [re.search(r"\[(.*?)\]", tit[i]).group(1) for i in sn],
    "RNA-seq, PD with dementia", sex=meta_sex([sx[i] for i in sn]))

# ---------------- GSE168496 (RNA-seq, transcript level; Netherlands Brain Bank IDs) ----------------
meta, _ = series_matrix("GSE168496")
tit = meta["!Sample_title"][0]; dx = chars(meta, "disease state")
T = pd.read_csv(fetch("GSE168496", "suppl", "GSE168496_all_samples_preprocessed_data.tsv.gz"), sep="\t", index_col=0)
T.index = T.index.astype(str).str.split(".").str[0]
T = T.groupby(level=0).sum()[tit]
def tx_to_gene(ids):
    conv = gconvert(ids, "ENST", batch=1500)
    return pd.DataFrame({g: T.loc[[t for t in ts if t in T.index]].sum(0).to_numpy(float) for g, ts in conv.items()
                         if any(t in T.index for t in ts)}, index=T.columns).T
GT = np.log2(tx_to_gene(GENES) + 1)
MT = np.log2(tx_to_gene(MARKERS + SEXG) + 1)
print(f"   GSE168496: {T.shape[0]:,} transcripts, {T.shape[1]} people; column sums {T.sum(0).min():,.0f}-{T.sum(0).max():,.0f}")
add("GSE168496", GT, MT, [0 if "control" in d.lower() else 1 for d in dx], tit, "RNA-seq, NBB", sex=meta_sex(chars(meta, "gender")))
COV = pd.DataFrame([{"cohort": n, "platform": c["note"], "people": len(c["y"]), "control": int((c["y"] == 0).sum()), "PD": int(c["y"].sum()),
                     "discovery_genes_measured": c["G"].shape[0], "share": c["G"].shape[0] / len(GENES),
                     "panel_genes_measured": int(np.isin(PANEL, c["G"].index).sum()),
                     "neuron_markers": int(np.isin(MARKERS, c["Mk"].index).sum())} for n, c in COH.items()])
COV.to_csv(OUT / "cohorts.csv", index=False); print(COV.round(3).to_string(index=False))

## 3. Donor overlap with the discovery studies

In [ ]:
def nbb_key(s):
    """Netherlands Brain Bank IDs written 'PD-03-43' or '2003-043' -> (year, number)."""
    m = re.search(r"(\d{2,4})-(\d{1,3})$", s)
    return None if not m else (int(m.group(1)) % 100, int(m.group(2)))
HARV = {re.sub(r"\D", "", k): (k, v) for k, v in DISC["GSE20141"].items()}
NBB = {nbb_key(k): (k, v) for k, v in DISC["GSE182622"].items()}
FAMILY = {"GSE20292": "harvard", "GSE20163": "harvard", "GSE20164": "harvard", "GSE49036": "nbb", "GSE168496": "nbb"}
DETAIL, OVER = [], []
for name, c in COH.items():
    if name in ("GSE20292", "GSE20163"):
        match = [HARV.get(re.sub(r"\D", "", d)) for d in c["donor"]]; basis = "donor number = GSE20141"
    elif name == "GSE168496":
        match = [NBB.get(nbb_key(d)) for d in c["donor"]]; basis = "brain-bank ID = GSE182622"
    else:
        match = [None] * len(c["donor"])
        basis = "no donor IDs in a comparable format" if name in FAMILY else "different brain bank"
    c["overlap"] = np.array([m is not None for m in match])
    c["strict"] = FAMILY.get(name, "none")
    for i, m in enumerate(match):
        if m is not None:
            DETAIL.append({"cohort": name, "external_id": c["donor"][i], "discovery_id": m[0],
                           "diagnosis_agrees": int(c["y"][i]) == m[1][0], "sex_metadata": c["sex_meta"][i],
                           "sex_expression": int(c["sex_expr"][i]), "sex_discovery_expression": m[1][1]})
    OVER.append({"cohort": name, "people": len(c["donor"]), "shared_with_discovery": int(c["overlap"].sum()), "basis": basis,
                 "strict_model_trained_without": " + ".join(RELATED.get(c["strict"], [])) or "(same as frozen)"})
# donors that appear in two external cohorts (GSE20163 and GSE20292) are counted once when cohorts are pooled
seen = {re.sub(r"\D", "", d) for d, o in zip(COH["GSE20292"]["donor"], COH["GSE20292"]["overlap"]) if not o}
for name, c in COH.items():
    c["pool"] = ~c["overlap"]
    if name == "GSE20163":
        c["pool"] &= np.array([re.sub(r"\D", "", d) not in seen for d in c["donor"]])
OV = pd.DataFrame(OVER); OV["also_in_another_external_cohort"] = [int((~c["pool"] & ~c["overlap"]).sum()) for c in COH.values()]
DET = pd.DataFrame(DETAIL)
OV.to_csv(OUT / "donor_overlap.csv", index=False); DET.to_csv(OUT / "donor_overlap_matches.csv", index=False)
print(OV.to_string(index=False)); print(); print(DET.to_string(index=False))

## 4. Every cohort: frozen and strictly independent models, neuron content, calls at the fixed thresholds

In [ ]:
zrow = lambda D: ((D.T - D.T.mean()) / D.T.std(ddof=1).clip(lower=0.05)).T
rng = np.random.default_rng(42)
def boot_ci(yv, s, n=4000):
    bs = [roc_auc_score(yv[i], s[i]) for i in (rng.integers(0, len(yv), len(yv)) for _ in range(n)) if len(set(yv[i])) == 2]
    return np.percentile(bs, 2.5), np.percentile(bs, 97.5)
def perm_p(yv, s, n=10000):
    a = roc_auc_score(yv, s); null = np.array([roc_auc_score(rng.permutation(yv), s) for _ in range(n)])
    return (np.sum(null >= a) + 1) / (n + 1)
def prepare(G, Mk):
    Z = zrow(G).reindex(GENES).fillna(0.0).T.to_numpy()
    return Z, np.apply_along_axis(rankdata, 1, Z) / Z.shape[1], zrow(Mk.loc[[m for m in MARKERS if m in Mk.index]]).mean(0).to_numpy()
def score(model, ZR): return model["rf"].predict_proba(model["pca"].transform(ZR))[:, 1]
loo = lambda F, yv: roc_auc_score(yv, cross_val_predict(LogisticRegression(), F, yv, cv=LeaveOneOut(), method="predict_proba")[:, 1])
def calls(yv, s, t):
    yh = (s > t).astype(int)
    return {"accuracy": accuracy_score(yv, yh), "sensitivity": recall_score(yv, yh), "specificity": recall_score(1 - yv, 1 - yh),
            "balanced_accuracy": balanced_accuracy_score(yv, yh)}
def pscore(P, Z): return np.mean([m.predict_proba(Z[:, P["genes"]])[:, 1] for m in P["rfs"]], axis=0)
MODELS = ["frozen", "strict", "panel", "panel_strict"]       # core classifier as reported / strictly independent; panel forest likewise
ROWS, PEOPLE, SC = [], [], {}
for name, c in COH.items():
    k = ~c["overlap"]; fam = c["strict"]
    Z, ZR, NEU = prepare(c["G"].loc[:, k], c["Mk"].loc[:, k]); yv = c["y"][k]
    S = {"frozen": (score(CORE, ZR), CORE["thr"]), "strict": (score(STRICT[fam], ZR), STRICT[fam]["thr"]),
         "panel": (pscore(PANELS["none"], Z), PANEL_THR), "panel_strict": (pscore(PANELS[fam], Z), PANELS[fam]["thr"])}
    SC[name] = dict(y=yv, neuron=NEU, Z=Z, pool=c["pool"][k], scores={m: s for m, (s, _) in S.items()}, thr={m: t for m, (_, t) in S.items()})
    for m, (s, t) in S.items():
        lo, hi = boot_ci(yv, s)
        resid = s - np.polyval(np.polyfit(NEU, s, 1), NEU)
        r = {"cohort": name, "model": m, "n": len(yv), "control": int((yv == 0).sum()), "PD": int(yv.sum()),
             "auc": roc_auc_score(yv, s), "ci_lo": lo, "ci_hi": hi, "perm_p": perm_p(yv, s),
             "auc_neuron_markers": roc_auc_score(yv, -NEU), "auc_neuron_removed": roc_auc_score(yv, resid),
             "rho_with_neuron": spearmanr(s, NEU)[0], "loo_neuron_only": loo(NEU[:, None], yv),
             "loo_neuron_plus_score": loo(np.c_[NEU, s], yv), "threshold_oob": t}
        for tn, tv in (("0.5", 0.5), ("oob", t)):
            r.update({f"{kk}@{tn}": v for kk, v in calls(yv, s, tv).items()})
        ROWS.append(r)
    PEOPLE.append(pd.DataFrame({"cohort": name, "donor": c["donor"][k], "y": yv, "stage": np.array(c["stage"])[k],
                                **{m: S[m][0] for m in MODELS}, "neuron_score": NEU,
                                "counted_in_pool": c["pool"][k]}))
RES = pd.DataFrame(ROWS); RES.to_csv(OUT / "external_multi_results.csv", index=False)
pd.concat(PEOPLE).to_csv(OUT / "external_multi_scores.csv", index=False)
print(RES[["cohort", "model", "n", "control", "PD", "auc", "ci_lo", "ci_hi", "perm_p", "auc_neuron_markers", "auc_neuron_removed",
           "loo_neuron_only", "loo_neuron_plus_score"]].round(3).to_string(index=False))
print(); print(RES[["cohort", "model"] + [c for c in RES.columns if "@" in c]].round(2).to_string(index=False))

In [ ]:
# ---------------- cohorts together (each donor once) ----------------
def fast_auc(yv, s):
    r = rankdata(s); n1 = int(yv.sum()); n0 = len(yv) - n1
    return (r[yv == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan
def resid(s, neu): return s - np.polyval(np.polyfit(neu, s, 1), neu)
def pool_set(names, n_boot=4000):
    # cohort AUCs averaged with weights = PD-control pairs; 95% CI by resampling people within each cohort
    P = {n: (SC[n]["y"][SC[n]["pool"]], SC[n]["neuron"][SC[n]["pool"]], {m: SC[n]["scores"][m][SC[n]["pool"]] for m in MODELS}) for n in names}
    Y = [P[n][0] for n in names]; W = np.array([v.sum() * (1 - v).sum() for v in Y], float)
    get = {"neuron": [-P[n][1] for n in names]}
    for m in MODELS:
        get[m] = [P[n][2][m] for n in names]
        get[m + "|neuron_removed"] = [resid(P[n][2][m], P[n][1]) for n in names]
    full = [np.arange(len(v)) for v in Y]
    boots = [[rng.integers(0, len(v), len(v)) for v in Y] for _ in range(n_boot)]
    def est(S_, I):
        a = np.array([fast_auc(Y[j][i], S_[j][i]) for j, i in enumerate(I)]); ok = ~np.isnan(a)
        return np.sum(W[ok] * a[ok]) / W[ok].sum()
    out = {"cohorts": names, "people": int(sum(len(v) for v in Y)), "control": int(sum((v == 0).sum() for v in Y)),
           "PD": int(sum(v.sum() for v in Y)), "auc": {}}
    for key, S_ in get.items():
        bs = [est(S_, I) for I in boots]
        out["auc"][key] = {"auc": est(S_, full), "ci_lo": np.percentile(bs, 2.5), "ci_hi": np.percentile(bs, 97.5)}
    yy = np.concatenate(Y)
    for tn in ("0.5", "oob"):
        out[f"calls@{tn}"] = {m: calls(yy, np.concatenate([(P[n][2][m] > (0.5 if tn == "0.5" else SC[n]["thr"][m])).astype(float)
                                                           for n in names]), 0.5) for m in MODELS}
    return out
NAMES = list(SC)
UNSEEN = [n for n in NAMES if n != "GSE7621"]         # GSE7621 was looked at in earlier versions of the project
POOL = {"unseen": pool_set(UNSEEN), "all": pool_set(NAMES)}
for k, v in POOL.items():
    print(f"== {k}: {len(v['cohorts'])} cohorts, {v['people']} people ({v['control']} control, {v['PD']} PD)")
    for key, a in v["auc"].items():
        print(f"   {key:30s} AUC {a['auc']:.3f} ({a['ci_lo']:.2f}-{a['ci_hi']:.2f})")
    print(pd.DataFrame(v["calls@oob"]).T.round(3).to_string())

## 5. Gene by gene: the Boruta panel across the external cohorts (fixed-effect inverse-variance mean)

In [ ]:
def hedges(v, yv):
    a, b = v[yv == 1], v[yv == 0]; n1, n0 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n0 - 1) * b.var(ddof=1)) / (n1 + n0 - 2)) + 1e-9
    g = (1 - 3 / (4 * (n1 + n0) - 9)) * (a.mean() - b.mean()) / sp
    return g, (n1 + n0) / (n1 * n0) + g ** 2 / (2 * (n1 + n0))
EFF = []
for name in NAMES:
    S = SC[name]; p = S["pool"]; Z, yv, NEU = S["Z"][p], S["y"][p], S["neuron"][p]
    nc = NEU - NEU.mean(); ZA = Z - np.outer(nc, (Z * nc[:, None]).sum(0) / (nc ** 2).sum())
    for g in PANEL:
        if g in COH[name]["G"].index:
            j = GENES.index(g); gr, vr = hedges(Z[:, j], yv); ga, va = hedges(ZA[:, j], yv)
            EFF.append({"cohort": name, "gene": g, "symbol": SYM.get(g, g), "g": gr, "v": vr, "g_adj": ga, "v_adj": va})
EFF = pd.DataFrame(EFF); EFF.to_csv(OUT / "external_multi_gene_by_cohort.csv", index=False)
def ivw(col, vcol):
    return EFF.groupby("gene").apply(lambda d: pd.Series({"g": np.sum(d[col] / d[vcol]) / np.sum(1 / d[vcol]),
                                                        "se": np.sqrt(1 / np.sum(1 / d[vcol])), "k": len(d)}))
MR, MA = ivw("g", "v").reindex(PANEL), ivw("g_adj", "v_adj").reindex(PANEL)
GM = pd.DataFrame({"symbol": [SYM.get(g, g) for g in PANEL], "g_lcm": DE.loc[PANEL, "hedges_g_meta"].to_numpy(),
                   "deg": DE.loc[PANEL, "is_deg"].to_numpy(), "g_external": MR["g"].to_numpy(), "se_external": MR["se"].to_numpy(),
                   "g_external_neuron_adj": MA["g"].to_numpy(), "se_external_neuron_adj": MA["se"].to_numpy(),
                   "k_cohorts": MR["k"].to_numpy()}, index=pd.Index(PANEL, name="gene"))
GM.to_csv(OUT / "external_multi_gene_meta.csv")
AGREE = {}
for col in ("g_external", "g_external_neuron_adj"):
    d = GM.dropna(subset=[col]); k = int((np.sign(d[col]) == np.sign(d.g_lcm)).sum())
    AGREE[col] = {"same_sign": k, "n": len(d), "binom_p": binomtest(k, len(d), 0.5, alternative="greater").pvalue,
                  "spearman": spearmanr(d.g_lcm, d[col])[0]}
print(GM.round(2).to_string()); print(json.dumps(AGREE, indent=1, default=float))

## 6. Early stage: incidental Lewy body disease (Braak 1-4) against controls, GSE49036

In [ ]:
e = EARLY_RAW; m = e["lab"] != 1
Z, ZR, NEU = prepare(e["G"].loc[:, m], e["Mk"].loc[:, m]); ye = (e["lab"][m] == 2).astype(int)
EARLY = {"control": int((ye == 0).sum()), "ilbd": int(ye.sum())}
for key, mod in (("frozen", CORE), ("strict", STRICT["nbb"]), ("panel", PANELS["none"]), ("panel_strict", PANELS["nbb"])):
    s = score(mod, ZR) if "pca" in mod else pscore(mod, Z); lo, hi = boot_ci(ye, s)
    EARLY[key] = {"auc": roc_auc_score(ye, s), "ci_lo": lo, "ci_hi": hi, "perm_p": perm_p(ye, s),
                  "auc_neuron_removed": roc_auc_score(ye, s - np.polyval(np.polyfit(NEU, s, 1), NEU)),
                  **{f"{k}@oob": v for k, v in calls(ye, s, mod["thr"]).items()}}
EARLY["auc_neuron_markers"] = roc_auc_score(ye, -NEU)
# score along the whole Braak sequence (all 28 GSE49036 donors, one within-cohort standardisation)
Z, ZR, NEU = prepare(e["G"], e["Mk"])
ST = pd.DataFrame({"donor": e["donor"], "braak": e["braak"], "frozen": score(CORE, ZR), "strict": score(STRICT["nbb"], ZR), "neuron_score": NEU})
ST["stage_rank"] = ST.braak.map({"CTRL": 0, "BR12": 1, "BR34": 2, "BR56": 3}); ST.to_csv(OUT / "gse49036_braak_scores.csv", index=False)
EARLY["spearman_strict_with_braak"] = spearmanr(ST.stage_rank, ST.strict)[0]
EARLY["spearman_neuron_with_braak"] = spearmanr(ST.stage_rank, ST.neuron_score)[0]
print(json.dumps(EARLY, indent=1, default=float)); print(ST.groupby("braak")[["frozen", "strict", "neuron_score"]].mean().round(3))
json.dump({"thresholds": THRESHOLDS, "strict_models": {k: {"trained_without": v, "people": STRICT[k]["n"], "components": STRICT[k]["components"],
                                                          "panel_genes": [SYM.get(GENES[j], GENES[j]) for j in PANELS[k]["genes"]],
                                                          "panel_genes_in_reported_panel": len(set(PANELS[k]["genes"]) & set(PI))}
                             for k, v in RELATED.items()},
           "pooled": POOL, "gene_agreement": AGREE, "early_stage_GSE49036": EARLY, "donor_overlap": OV.to_dict("records"),
           "donor_matches": DET.to_dict("records")}, open(OUT / "external_multi_summary.json", "w"), indent=1, default=float)
import shutil; shutil.rmtree(GEO, ignore_errors=True)
log("analysis done")

## 7. The figure (print size, 183 x 150 mm)

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle

FIG_IN = Path(os.environ.get("FIG_IN", "/kaggle/working")); FIG_OUT = Path(os.environ.get("FIG_OUT", "/kaggle/working"))
R = pd.read_csv(FIG_IN / "external_multi_results.csv").set_index(["cohort", "model"])
OVR = pd.read_csv(FIG_IN / "donor_overlap.csv").set_index("cohort")
SUMM = json.load(open(FIG_IN / "external_multi_summary.json"))
GMF = pd.read_csv(FIG_IN / "external_multi_gene_meta.csv")

INK, MUTED, RULE = "#1B1D20", "#5E656D", "#C9CED4"
CORE_C, PANEL_C, NEURO_C = "#23324A", "#8E1B2E", "#9AA1A9"
SETC = {True: "#8E1B2E", False: "#2E5A87"}
FONT_STACK = ["Helvetica Neue", "Helvetica", "Arial", "Liberation Sans", "Nimbus Sans", "FreeSans", "DejaVu Sans"]
plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": FONT_STACK, "font.size": 6.3,
                     "axes.linewidth": 0.5, "axes.edgecolor": "#30343A", "axes.labelcolor": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "xtick.labelsize": 5.8, "ytick.labelsize": 5.8,
                     "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2.0, "ytick.major.size": 2.0,
                     "xtick.major.pad": 1.6, "ytick.major.pad": 1.6, "xtick.color": "#30343A", "ytick.color": "#30343A",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 150, "savefig.facecolor": "white", "figure.facecolor": "white",
                     "mathtext.fontset": "custom", "mathtext.rm": "sans", "mathtext.it": "sans:italic"})

FW, FH = 183.0, 150.0
fig = plt.figure(figsize=(FW / 25.4, FH / 25.4))
M = fig.add_axes([0, 0, 1, 1]); M.set_xlim(0, FW); M.set_ylim(0, FH); M.axis("off")
def fax(x, y_, w, h): return fig.add_axes([x / FW, y_ / FH, w / FW, h / FH])
def letter(x, y_, L, title):
    M.text(x, y_, L, ha="left", va="baseline", fontsize=9, fontweight="bold", color=INK)
    M.text(x + 4.2, y_, title, ha="left", va="baseline", fontsize=7.2, color=INK)
def rule(x0, x1, y_, lw=0.4, color=RULE): M.plot([x0, x1], [y_, y_], color=color, lw=lw, solid_capstyle="butt")
def mark(x, y_, color, filled=True, size=11, marker="o"):
    M.scatter([x], [y_], s=size, marker=marker, facecolor=color if filled else "white", edgecolor=color, linewidth=0.8, zorder=5, clip_on=False)
f2 = lambda v: f"{v:.2f}"

# ======================= a: forest plot =======================
POOL_U, POOL_A = SUMM["pooled"]["unseen"], SUMM["pooled"]["all"]
ORDER = ["GSE20292", "GSE20163", "GSE20164", "GSE8397", "GSE49036", "GSE114517", "GSE168496", "GSE7621"]
PLAT = {"GSE20292": "array (U133A)", "GSE20163": "array (U133A)", "GSE20164": "array (U133A)", "GSE8397": "array (U133A)",
        "GSE49036": "array (U133 Plus 2)", "GSE114517": "RNA-seq", "GSE168496": "RNA-seq", "GSE7621": "array (U133 Plus 2)"}
letter(2.0, 145.0, "a", "The frozen models in eight independent bulk substantia nigra cohorts")
Y0, STEP = 131.0, 4.9
ROWY = {c: Y0 - i * STEP for i, c in enumerate(ORDER)}
ROWY["GSE7621"] -= 1.2
PY = {"unseen": ROWY["GSE7621"] - 7.0, "all": ROWY["GSE7621"] - 7.0 - STEP}
YB, YT = PY["all"] - 3.2, Y0 + 3.2
AX = {"core": (76.0, 34.0), "panel": (128.0, 34.0)}
NUMX = {"core": (114.0, 120.0), "panel": (166.0, 172.0)}
XNEU = 177.5

# column headers
HY = YT + 2.4
for x, t, ha in ((4.0, "cohort", "left"), (21.0, "platform", "left"), (52.5, "control / PD", "center"), (65.5, "shared donors\nremoved", "center")):
    M.text(x, HY, t, ha=ha, va="bottom", fontsize=5.9, color=MUTED, linespacing=1.05)
for key, (x0, w) in AX.items():
    M.text(x0 + w / 2, HY + 3.4, "core classifier" if key == "core" else "Boruta-panel forest", ha="center", va="bottom",
           fontsize=6.6, color=CORE_C if key == "core" else PANEL_C, fontweight="bold")
    M.text(x0 + w / 2, HY, "AUC", ha="center", va="bottom", fontsize=5.9, color=MUTED)
    c = CORE_C if key == "core" else PANEL_C
    mark(NUMX[key][0], HY + 1.0, c, True, 9); mark(NUMX[key][1], HY + 1.0, c, False, 9)
M.scatter([XNEU], [HY + 1.0], s=16, marker="|", color=NEURO_C, linewidth=1.1, clip_on=False)
M.text(XNEU, HY + 3.0, "neuron\nmarkers", ha="center", va="bottom", fontsize=5.6, color=MUTED, linespacing=1.0)
rule(2.0, 181.0, YT + 0.9, lw=0.6, color="#30343A")

# forest axes
AXES = {}
for key, (x0, w) in AX.items():
    ax = fax(x0, YB, w, YT - YB); ax.set_xlim(0.12, 1.02); ax.set_ylim(YB, YT)
    ax.spines["left"].set_visible(False); ax.set_yticks([])
    ax.set_xticks([0.25, 0.5, 0.75, 1.0]); ax.set_xticklabels(["0.25", "0.5", "0.75", "1"])
    ax.axvline(0.5, color="#B7BDC4", lw=0.5, ls=(0, (2, 2)), zorder=0)
    ax.patch.set_alpha(0)
    AXES[key] = ax
for i, c in enumerate(ORDER[:-1]):                                   # light zebra rows across the whole table
    if i % 2 == 0:
        M.add_patch(Rectangle((2.0, ROWY[c] - STEP / 2), 179.0, STEP, color="#F5F4F1", lw=0, zorder=0))

def cohort_row(c, yy):
    r = R.loc[(c, "frozen")]
    name = c + ("$^{\\dagger}$" if c == "GSE7621" else "")
    M.text(4.0, yy, name, ha="left", va="center", fontsize=6.2)
    M.text(21.0, yy, PLAT[c], ha="left", va="center", fontsize=5.9, color=MUTED)
    M.text(52.5, yy, f"{int(r.control)} / {int(r.PD)}", ha="center", va="center", fontsize=6.0)
    k = int(OVR.loc[c, "shared_with_discovery"])
    M.text(65.5, yy, str(k) if k else "–", ha="center", va="center", fontsize=6.0, color=INK if k else MUTED,
           fontweight="bold" if k else "normal")
    for key, (m_rep, m_str) in (("core", ("frozen", "strict")), ("panel", ("panel", "panel_strict"))):
        ax = AXES[key]; col = CORE_C if key == "core" else PANEL_C
        a, s_ = R.loc[(c, m_rep)], R.loc[(c, m_str)]
        ax.plot([a.ci_lo, a.ci_hi], [yy, yy], color=col, lw=0.8, solid_capstyle="butt", zorder=2)
        ax.scatter([a.auc_neuron_markers], [yy], s=22, marker="|", color=NEURO_C, linewidth=1.1, zorder=3)
        ax.scatter([s_.auc], [yy], s=13, facecolor="white", edgecolor=col, linewidth=0.8, zorder=4)
        ax.scatter([a.auc], [yy], s=13, color=col, zorder=5, linewidth=0)
        M.text(NUMX[key][0], yy, f2(a.auc), ha="center", va="center", fontsize=6.0, color=INK)
        M.text(NUMX[key][1], yy, f2(s_.auc), ha="center", va="center", fontsize=6.0, color=MUTED)
    M.text(XNEU, yy, f2(r.auc_neuron_markers), ha="center", va="center", fontsize=6.0, color=MUTED)
for c in ORDER:
    cohort_row(c, ROWY[c])
rule(2.0, 181.0, ROWY["GSE7621"] + STEP / 2 + 0.25, lw=0.3)

# pooled rows
rule(2.0, 181.0, PY["unseen"] + STEP / 2 + 0.6, lw=0.5, color="#30343A")
for tag, P in (("unseen", POOL_U), ("all", POOL_A)):
    yy = PY[tag]
    lab = f"pooled, {len(P['cohorts'])} unseen cohorts" if tag == "unseen" else f"pooled, all {len(P['cohorts'])} cohorts"
    M.text(4.0, yy, lab, ha="left", va="center", fontsize=6.2, fontweight="bold" if tag == "unseen" else "normal")
    M.text(52.5, yy, f"{P['control']} / {P['PD']}", ha="center", va="center", fontsize=6.0,
           fontweight="bold" if tag == "unseen" else "normal")
    for key, (m_rep, m_str) in (("core", ("frozen", "strict")), ("panel", ("panel", "panel_strict"))):
        ax = AXES[key]; col = CORE_C if key == "core" else PANEL_C
        a, s_ = P["auc"][m_rep], P["auc"][m_str]
        h = 1.45
        ax.add_patch(Polygon([[a["ci_lo"], yy], [a["auc"], yy + h], [a["ci_hi"], yy], [a["auc"], yy - h]], closed=True,
                             facecolor=col, alpha=1.0 if tag == "unseen" else 0.45, edgecolor="none", zorder=4))
        ax.scatter([s_["auc"]], [yy], s=13, facecolor="white", edgecolor=col, linewidth=0.8, zorder=5)
        ax.scatter([P["auc"]["neuron"]["auc"]], [yy], s=22, marker="|", color=NEURO_C, linewidth=1.1, zorder=3)
        M.text(NUMX[key][0], yy, f2(a["auc"]), ha="center", va="center", fontsize=6.0, fontweight="bold" if tag == "unseen" else "normal")
        M.text(NUMX[key][1], yy, f2(s_["auc"]), ha="center", va="center", fontsize=6.0, color=MUTED)
    M.text(XNEU, yy, f2(P["auc"]["neuron"]["auc"]), ha="center", va="center", fontsize=6.0, color=MUTED)
for key, (x0, w) in AX.items():
    M.text(x0 + w / 2, YB - 5.4, "AUC in the external cohort", ha="center", va="top", fontsize=6.0, color=INK)

# key and footnote
KY = YB - 11.3
mark(4.8, KY, "#4A4F56", True, 11); M.text(6.8, KY, "model as reported (trained on all 63 discovery people), with 95% CI", va="center", fontsize=5.9)
mark(84.8, KY, "#4A4F56", False, 11)
M.text(86.8, KY, "strictly independent: same recipe, retrained without any discovery study from the same brain bank",
       va="center", fontsize=5.9)
M.scatter([4.8], [KY - 3.6], s=22, marker="|", color=NEURO_C, linewidth=1.1)
M.text(6.8, KY - 3.6, "8 dopamine-neuron marker genes alone (fewer neurons = more PD-like)", va="center", fontsize=5.9)
M.text(84.0, KY - 3.6, "$^{\\dagger}$examined in earlier versions of this project; the other seven were first opened after the model was locked",
       va="center", fontsize=5.6, color=MUTED)

# ======================= b: pooled table =======================
TOPB = KY - 8.8
rule(2.0, 181.0, TOPB + 2.2, lw=0.4)
U = POOL_U
letter(2.0, TOPB - 3.2, "b", f"Pooled over the {len(U['cohorts'])} unseen cohorts ({U['people']} people)")
T0 = TOPB - 7.6
COLS = [(62.0, "AUC (95% CI)"), (82.0, "neuron content\nremoved"), (95.5, "accuracy"), (107.5, "sensitivity"), (119.5, "specificity")]
rule(4.0, 125.0, T0, lw=0.6, color="#30343A")
for x, t in COLS:
    M.text(x, T0 - 1.2, t, ha="center", va="top", fontsize=5.9, color=MUTED, linespacing=1.0)
rule(4.0, 125.0, T0 - 6.0, lw=0.4)
TROWS = [("neuron", "dopamine-neuron markers alone", NEURO_C, None),
         ("frozen", "core classifier", CORE_C, True), ("strict", "   strictly independent", CORE_C, False),
         ("panel", "Boruta-panel forest", PANEL_C, True), ("panel_strict", "   strictly independent", PANEL_C, False)]
CALLS = U["calls@oob"]
for i, (key, lab, col, filled) in enumerate(TROWS):
    yy = T0 - 9.4 - i * 4.3
    if filled is None:
        M.scatter([6.0], [yy], s=20, marker="|", color=col, linewidth=1.1)
    else:
        mark(6.0, yy, col, filled, 10)
    bold = key in ("frozen", "panel")
    M.text(8.5, yy, lab.strip() if not lab.startswith(" ") else "strictly independent", ha="left", va="center", fontsize=6.1,
           color=INK if bold or key == "neuron" else MUTED, fontweight="bold" if key == "panel" else "normal")
    a = U["auc"][key]
    M.text(COLS[0][0], yy, f"{a['auc']:.2f} ({a['ci_lo']:.2f}–{a['ci_hi']:.2f})", ha="center", va="center", fontsize=6.1,
           fontweight="bold" if key == "panel" else "normal")
    if key == "neuron":
        for x, _ in COLS[1:]:
            M.text(x, yy, "–", ha="center", va="center", fontsize=6.0, color=MUTED)
        continue
    nr = U["auc"][key + "|neuron_removed"]
    M.text(COLS[1][0], yy, f2(nr["auc"]), ha="center", va="center", fontsize=6.1, fontweight="bold" if key == "panel" else "normal")
    for (x, _), k in zip(COLS[2:], ("accuracy", "sensitivity", "specificity")):
        M.text(x, yy, f2(CALLS[key][k]), ha="center", va="center", fontsize=6.1)
ybot = T0 - 9.4 - 4 * 4.3 - 2.6
rule(4.0, 125.0, ybot, lw=0.6, color="#30343A")
TH = SUMM["thresholds"]
M.text(4.0, ybot - 2.6, "Accuracy, sensitivity and specificity at each forest's out-of-bag cut-off, fixed on the discovery people before any "
       "external data were seen", ha="left", va="top", fontsize=5.5, color=MUTED)
M.text(4.0, ybot - 5.4, f"(core {TH['core']:.2f}, panel {TH['panel']:.2f}). Cohort AUCs are weighted by PD-control pairs; "
       "95% CI by resampling people within cohorts.", ha="left", va="top", fontsize=5.5, color=MUTED)
M.text(4.0, ybot - 8.2, "Neuron content removed: each score regressed on the 8-gene neuron score within its cohort.",
       ha="left", va="top", fontsize=5.5, color=MUTED)

# ======================= c: gene-level agreement =======================
letter(131.0, TOPB - 3.2, "c", "The panel genes, gene by gene")
G = GMF.copy()
axC = fax(140.0, 8.5, 39.0, TOPB - 17.5)
lim = 1.45
axC.set_xlim(-lim, lim); axC.set_ylim(-lim * 0.9, lim * 0.9)
axC.add_patch(Rectangle((0, 0), lim, lim, color="#F2EFEA", lw=0, zorder=0))
axC.add_patch(Rectangle((-lim, -lim), lim, lim, color="#F2EFEA", lw=0, zorder=0))
axC.axhline(0, color="#9AA1A9", lw=0.4, zorder=1); axC.axvline(0, color="#9AA1A9", lw=0.4, zorder=1)
for r in G.itertuples():
    axC.plot([r.g_lcm, r.g_lcm], [r.g_external, r.g_external_neuron_adj], color="#B9BEC4", lw=0.45, zorder=2)
axC.scatter(G.g_lcm, G.g_external, s=7, facecolor="white", edgecolor="#9AA1A9", linewidth=0.5, zorder=3)
axC.scatter(G.g_lcm, G.g_external_neuron_adj, s=12, color=[SETC[bool(d)] for d in G.deg], linewidth=0, zorder=4)
axC.set_xticks([-1, 0, 1]); axC.set_yticks([-1, 0, 1])
axC.set_xlabel("Hedges' g, laser-capture neurons", fontsize=6.0, labelpad=1.5)
axC.set_ylabel("Hedges' g, bulk nigra (pooled)", fontsize=6.0, labelpad=1.0)
AG = SUMM["gene_agreement"]; ga, gr = AG["g_external_neuron_adj"], AG["g_external"]
pfmt = lambda p: f"{p:.3f}" if p >= 0.001 else f"{p:.0e}"
axC.text(-lim + 0.07, lim * 0.9 - 0.07, f"{ga['same_sign']}/{ga['n']} same sign\nP = {pfmt(ga['binom_p'])}, ρ = {ga['spearman']:.2f}",
         ha="left", va="top", fontsize=5.6, color=INK, linespacing=1.15)
axC.text(lim - 0.07, -lim * 0.9 + 0.07, f"unadjusted\n{gr['same_sign']}/{gr['n']}, ρ = {gr['spearman']:.2f}",
         ha="right", va="bottom", fontsize=5.4, color=MUTED, linespacing=1.15)
# key for c, under the title
KX, KC = 131.5, TOPB - 8.0
axK = [(SETC[True], True, "Boruta & DEG"), (SETC[False], True, "Boruta only"), ("#9AA1A9", False, "unadjusted")]
xx = KX
for col, filled, t in axK:
    M.scatter([xx + 0.8], [KC], s=9 if filled else 7, facecolor=col if filled else "white", edgecolor=col, linewidth=0.6)
    M.text(xx + 2.3, KC, t, va="center", fontsize=5.6, color=MUTED)
    xx += 2.3 + len(t) * 1.02 + 2.6

for ext in ("pdf", "png"):
    fig.savefig(FIG_OUT / f"Figure07_external_multicohort.{ext}", dpi=600 if ext == "png" else None)
print("saved Figure07_external_multicohort")


## 8. Ready-to-paste text: legend, methods, numbers

In [ ]:
U, A = POOL["unseen"], POOL["all"]
def ci(d): return f"{d['auc']:.2f} (95% CI {d['ci_lo']:.2f}-{d['ci_hi']:.2f})"
cu = U["calls@oob"]
OVI = OV.set_index("cohort"); r1, r2 = OVI.loc["GSE20292"], OVI.loc["GSE20163"]
ga, gr = AGREE["g_external_neuron_adj"], AGREE["g_external"]
TEXT = f"""FIGURE LEGEND
External validation in eight independent bulk substantia nigra cohorts. Both models were frozen before these cohorts
were opened (GSE7621, marked with a dagger, had been examined in earlier versions of the project) and were applied without
refitting; each cohort's genes were z-scored within the cohort and ranked within each person, exactly as in discovery.
(a) AUC of the core classifier and of the Boruta-panel forest in each cohort. Filled circle, model as reported, with 95%
bootstrap CI; open circle, strictly independent version of the same recipe, retrained without any discovery study that
could share donors with that cohort's brain bank; grey tick, the eight dopamine-neuron marker genes alone. Donors whose
brain-bank IDs matched a discovery donor were removed before testing (GSE20292, {r1.shared_with_discovery} of {r1.people};
GSE20163, {r2.shared_with_discovery} of {r2.people}; diagnosis agreed for every match). Pooled rows: cohort AUCs weighted by the number of
PD-control pairs, each donor counted once; diamond width, 95% CI. (b) Pooled over the {len(U['cohorts'])} cohorts never examined before the
model was locked ({U['people']} people): AUC; AUC after the neuron-marker score was regressed out of each model's score within each
cohort; and accuracy, sensitivity and specificity at the out-of-bag cut-off fixed on the 63 discovery people. (c) PD-versus-
control effect (Hedges' g) of each of the {ga['n']} panel genes in the laser-capture discovery cohort and, pooled by inverse variance, in the
eight bulk cohorts after adjustment for neuron content (filled) and without it (open). Shaded quadrants, same direction.

METHODS - External validation
Model lock. The core classifier (within-person gene ranks, 30 principal components, Random Forest of 1,000 trees) was chosen by
a sweep over 97 Random Forest variants that used only the 63 discovery people; the Boruta panel and its forest were fitted
on the same 63 people. Both were frozen, together with their decision thresholds, before any external cohort was analysed.
Cohorts. Eight public bulk substantia nigra cohorts were downloaded from GEO: GSE7621, GSE20292, GSE20163, GSE20164 and GSE8397
(Affymetrix HG-U133A or Plus 2; for GSE8397 the lateral and medial nigra of each case were averaged and frontal cortex excluded),
GSE49036 (HG-U133 Plus 2; controls and Braak 3-6 PD, with incidental Lewy body cases analysed separately), GSE114517 (RNA-seq
counts, nigra only; PD with dementia) and GSE168496 (RNA-seq; transcript abundances summed to genes). Probe sets were mapped to
Ensembl genes with g:Profiler (the highest-mean probe set per gene) and array data were log2-transformed. Each cohort was
processed without its labels: every gene was z-scored within the cohort and ranked within each person; genes not measured
were set to the cohort mean (coverage {COV.share.min():.0%}-{COV.share.max():.0%} of the {len(GENES):,} discovery genes).
Donor overlap. Where external donor IDs were public in a comparable format they were matched to the discovery donors
(GSE20292 and GSE20163 against GSE20141; GSE168496 against GSE182622). Matched donors ({int(OV.shared_with_discovery.sum())} in total) were removed
before testing; every match agreed in diagnosis. One GSE20163 donor also present in GSE20292 was counted once in pooled
analyses. Because not every cohort reports donor IDs, each cohort was also scored by strictly independent models: the same
recipe (for the panel, Boruta itself) re-run without every discovery study from a potentially shared brain bank (without
GSE20141 and GSE24378 for the Harvard-series cohorts, {STRICT['harvard']['n']} people; without GSE182622 for the Netherlands Brain Bank
cohorts, {STRICT['nbb']['n']} people).
Thresholds and statistics. Two thresholds were fixed on the discovery people before testing: 0.5, and each forest's out-of-bag
accuracy optimum (core {THRESHOLDS['core']:.2f}; panel {THRESHOLDS['panel']:.2f}). AUC 95% CIs came from 4,000 bootstrap resamples of people and
P values from 10,000 label permutations within each cohort. Pooled AUCs were cohort AUCs weighted by the number of PD-control
pairs, with CIs from resampling people within cohorts. Neuron content was the mean z-score of eight dopamine-neuron marker
genes (TH, SLC6A3, SLC18A2, DDC, KCNJ6, ALDH1A1, NR4A2, EN1); neuron-adjusted AUCs used the residual of each score after linear
regression on this estimate within the cohort. Gene-level effects were Hedges' g (PD minus control), pooled across cohorts
by inverse-variance weighting, with and without the neuron estimate regressed out of each gene.

RESULTS - numbers
Seven unseen cohorts ({U['people']} people, {U['control']} control, {U['PD']} PD):
  core classifier       AUC {ci(U['auc']['frozen'])}; neuron-adjusted {U['auc']['frozen|neuron_removed']['auc']:.2f}; strictly independent {U['auc']['strict']['auc']:.2f}
  Boruta-panel forest   AUC {ci(U['auc']['panel'])}; neuron-adjusted {ci(U['auc']['panel|neuron_removed'])}
                        strictly independent {ci(U['auc']['panel_strict'])}
  neuron markers alone  AUC {ci(U['auc']['neuron'])}
  at the discovery cut-off: core accuracy {cu['frozen']['accuracy']:.2f}, sensitivity {cu['frozen']['sensitivity']:.2f}, specificity {cu['frozen']['specificity']:.2f}
                            panel accuracy {cu['panel']['accuracy']:.2f}, sensitivity {cu['panel']['sensitivity']:.2f}, specificity {cu['panel']['specificity']:.2f}
All eight cohorts ({A['people']} people): core {ci(A['auc']['frozen'])}; panel {ci(A['auc']['panel'])}; neuron markers {ci(A['auc']['neuron'])}
Gene level: {ga['same_sign']}/{ga['n']} panel genes in the discovery direction after neuron adjustment (binomial P = {ga['binom_p']:.4f},
  Spearman {ga['spearman']:.2f}); {gr['same_sign']}/{gr['n']} without adjustment.
Early stage (GSE49036, {EARLY['ilbd']} incidental Lewy body vs {EARLY['control']} control): core {EARLY['frozen']['auc']:.2f}, core strict {EARLY['strict']['auc']:.2f},
  panel {EARLY['panel']['auc']:.2f}, panel strict {EARLY['panel_strict']['auc']:.2f}, neuron markers {EARLY['auc_neuron_markers']:.2f}
"""
print(TEXT)
open(OUT / "external_multi_legend_methods.txt", "w").write(TEXT)
